In [ ]:
#read the parquet file
import copy
from dataclasses import dataclass

import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import brentq
from scipy.interpolate import PchipInterpolator


In [ ]:
df = pd.read_parquet("unpivoted_bid_offer_train_clean.parquet", engine="pyarrow")

print(df.head())
print(df.info())
print(df.describe())
print(df.dtypes)


In [ ]:
if df is not None:
    print(df[["future_contract_code", "expiration"]].drop_duplicates().sort_values("expiration").head(6))

In [ ]:
def calculate_d1_d2(f, K, sigma, T,t):
    d1 = (np.log(f / K) + 0.5*sigma ** 2 * (T-t)) / (sigma * np.sqrt(T-t))
    d2 = d1 - sigma * np.sqrt(T-t)
    return d1,d2


def black_76(F, K, sigma, t, T, option_type='call'):
    d1, d2 = calculate_d1_d2(F, K, sigma,T, t)

    if option_type == 'call':
        return F * norm.cdf(d1) - K * norm.cdf(d2)

    elif option_type == 'put':
        return K * norm.cdf(-d2) - F * norm.cdf(-d1)

    else:
        raise ValueError("Invalid option type")

def black_delta(F, K, sigma, tau, option_type='call'):
    d1, _ = calculate_d1_d2(F, K, sigma, tau, 0.0)
    return norm.cdf(d1) if option_type == 'call' else norm.cdf(d1) - 1.0


def black_vega(F, K, sigma, tau):
    d1, _ = calculate_d1_d2(F, K, sigma, tau, 0.0)
    return F * norm.pdf(d1) * np.sqrt(tau)


def implied_volatility(f, K, T, t, market_price, option_type='call', tol=1e-8, max_iterations=200):
    if T - t <= 0:
        return np.nan
    intrinsic = max(f - K, 0.0) if option_type == 'call' else max(K - f, 0.0)
    upper = f if option_type == 'call' else K
    if not (intrinsic < market_price < upper):
        return np.nan
    objective_function = lambda sigma: black_76(f, K, sigma, t, T, option_type) - market_price
    lo, hi = 1e-6, 5.0
    if objective_function(lo) * objective_function(hi) > 0:
        return np.nan
    return brentq(objective_function, lo, hi, xtol=tol, maxiter=max_iterations)

def _black_on_forward(Fwd, K, v, D, option_type):
    d1 = (np.log(Fwd / K) + 0.5 * v ** 2) / v
    d2 = d1 - v
    if option_type == 'call':
        return D * (Fwd * norm.cdf(d1) - K * norm.cdf(d2))
    return D * (K * norm.cdf(-d2) - Fwd * norm.cdf(-d1))


In [ ]:
class Smile:
    def __init__(self, strikes, ivs, F, shift=0.0):
        strikes = np.asarray(strikes, float)
        ivs = np.asarray(ivs, float)
        ok = np.isfinite(strikes) & np.isfinite(ivs) & (ivs > 0)
        x = np.log(strikes[ok] / F)
        v = ivs[ok]
        if len(x) == 0:
            raise ValueError("Aucun point de smile valide.")
        ux, inv = np.unique(x, return_inverse=True)
        uv = np.bincount(inv, weights=v) / np.bincount(inv)
        self.x, self.v, self.F, self.shift = ux, uv, F, shift
        self._interp = PchipInterpolator(ux, uv) if len(ux) >= 2 else None

    def __call__(self, K):
        x = np.log(K / self.F)
        if self._interp is None:
            base = float(self.v[0])
        else:
            base = float(self._interp(np.clip(x, self.x[0], self.x[-1])))
        return max(base + self.shift, 1e-4)

    def shifted(self, dvol):
        new = copy.copy(self)
        new.shift = self.shift + dvol
        return new


def smile_from_quotes(q_day, F, tau, max_spread_ratio=0.6, min_price=0.02):
    # q_day : quotes d'un seul (date, contrat) avec colonnes opt_type, strike, bid, ask, mid.
    x = q_day.dropna(subset=["bid", "ask"]).copy()
    x = x[(x["bid"] > 0) & (x["ask"] > x["bid"])]
    x = x[(x["ask"] - x["bid"]) / x["ask"] <= max_spread_ratio]
    x = x[x["mid"] >= min_price]
    otm = ((x["strike"] >= F) & (x["opt_type"] == "call")) | ((x["strike"] < F) & (x["opt_type"] == "put"))
    x = x[otm]
    x["iv"] = [implied_volatility(F, K, tau, 0.0, p, ot)
               for K, p, ot in zip(x["strike"], x["mid"], x["opt_type"])]
    x = x.dropna(subset=["iv"])
    if len(x) < 3:
        return None, None  # trop peu de points : a traiter (reprendre la veille, par exemple)
    return Smile(x["strike"].values, x["iv"].values, F), x[["strike", "opt_type", "iv"]]

In [ ]:
@dataclass
class Structure:
    fix_dates: pd.DatetimeIndex
    leg: np.ndarray  # 1 = first contract (Apr21), 2 = second (May21), by fixing date
    notional: float = 1_000_000.0
    K_call: float = 78.0
    K_put: float = np.nan

    @property
    def N(self):
        return len(self.fix_dates)


def build_structure(year, month, front_expiry, holidays=(), notional=1_000_000.0, K_call=78.0):
    start = pd.Timestamp(year, month, 1)
    end = start + pd.offsets.MonthEnd(0)
    days = pd.bdate_range(start, end)
    hol = [pd.Timestamp(h) for h in holidays]
    days = days[~days.isin(hol)]
    before = days[days <= pd.Timestamp(front_expiry)]
    d_star = before[-2]
    leg = np.where(days <= d_star, 1, 2)
    return Structure(days, leg, notional, K_call)


struct = build_structure(2021, 2, front_expiry="2021-02-26")
print("N =", struct.N, "| n1 =", int((struct.leg == 1).sum()), "| n2 =", int((struct.leg == 2).sum()))

In [ ]:
def _chol(S):
    jitter = 0.0
    scale = np.trace(S) / len(S)
    for _ in range(8):
        try:
            return np.linalg.cholesky(S + jitter * scale * np.eye(len(S)))
        except np.linalg.LinAlgError:
            jitter = 1e-14 if jitter == 0 else jitter * 100
    raise np.linalg.LinAlgError("Covariance not defined positive.")


def lognormal_cov(sig1, sig2, rho, theta, leg):
    # Sigma_ab = sig_a sig_b corr_ab min(theta_a, theta_b)
    sig = np.where(leg == 1, sig1, sig2)
    corr = np.where(leg[:, None] == leg[None, :], 1.0, rho)
    return np.outer(sig, sig) * corr * np.minimum.outer(theta, theta)


def simulate_correlated_futures(f1, f2, sig1, sig2, rho, theta, leg, M=100_000, seed=0, antithetic=True):
    theta = np.asarray(theta, float)
    leg = np.asarray(leg)
    F = np.where(leg == 1, f1, f2)
    sig = np.where(leg == 1, sig1, sig2)
    mu = -0.5 * sig ** 2 * theta
    Sigma = lognormal_cov(sig1, sig2, rho, theta, leg)
    L = _chol(Sigma)
    rng = np.random.default_rng(seed)
    n = len(theta)
    if antithetic:
        Z = rng.standard_normal((M // 2, n))
        Z = np.vstack([Z, -Z])
    else:
        Z = rng.standard_normal((M, n))
    eta = mu + Z @ L.T
    return F * np.exp(eta), dict(F=F, mu=mu, Sigma=Sigma)


def basket_moments(F, a, Sigma):
    m1 = float(a @ F)
    cov = np.outer(F, F) * np.expm1(Sigma)
    return m1, float(a @ cov @ a)

def _split_fixings(struct, t, past_fix):
    t = pd.Timestamp(t)
    rem = np.asarray(struct.fix_dates > t)
    past_sum = 0.0
    for d in struct.fix_dates[~rem]:
        if past_fix is None or d not in past_fix:
            raise ValueError(f"Fixing missing for {d.date()}")
        past_sum += past_fix[d]
    return rem, past_sum


def basket_forward(struct, t, f1, f2, past_fix=None):
    rem, past_sum = _split_fixings(struct, t, past_fix)
    F = np.where(struct.leg[rem] == 1, f1, f2)
    return past_sum / struct.N + F.sum() / struct.N

def price_asian_basket(t, struct, K, f1, f2, smile1, smile2, rho, option_type='call',
                       method='mm', past_fix=None, D=1.0, M=200_000, seed=0,
                       antithetic=True, control=True, vols=None, return_se=False):
    t = pd.Timestamp(t)
    rem, past_sum = _split_fixings(struct, t, past_fix)
    N = struct.N
    n_rem = int(rem.sum())
    if n_rem == 0:                                           # tout est deja fixe
        A = past_sum / N
        pay = max(A - K, 0.0) if option_type == 'call' else max(K - A, 0.0)
        return (D * pay, 0.0) if return_se else D * pay

    leg = struct.leg[rem]
    theta = np.asarray((struct.fix_dates[rem] - t).days, float) / 365.0
    F = np.where(leg == 1, f1, f2)
    a = np.full(n_rem, 1.0 / N)
    Fbar = past_sum / N + float(a @ F)                       # forward of the full basket
    if vols is None:
        sig1 = smile1(f1 * K / Fbar)                         # equivalent strikes  (common moneyness)
        sig2 = smile2(f2 * K / Fbar)
    else:
        sig1, sig2 = vols
    Keff = K - past_sum / N
    Sigma = lognormal_cov(sig1, sig2, rho, theta, leg)
    m1, var = basket_moments(F, a, Sigma)

    if Keff <= 0:                                            # always in the money (call) / without value (put)
        pay = D * (m1 - Keff) if option_type == 'call' else 0.0
        return (pay, 0.0) if return_se else pay

    if method == 'mm':
        v = np.sqrt(np.log1p(var / m1 ** 2))
        price = _black_on_forward(m1, Keff, v, D, option_type)
        return (price, 0.0) if return_se else price

    # Monte Carlo
    fix, info = simulate_correlated_futures(f1, f2, sig1, sig2, rho, theta, leg, M, seed, antithetic)
    A_rem = fix @ a
    pay_a = np.maximum(A_rem - Keff, 0.0) if option_type == 'call' else np.maximum(Keff - A_rem, 0.0)
    if control:
        s = a.sum()
        at = a / s
        mG = float(at @ (np.log(F) + info["mu"]))
        vG2 = float(at @ info["Sigma"] @ at)
        Fgeo = s * np.exp(mG + 0.5 * vG2)
        G = s * np.exp(np.log(fix) @ at)
        pay_g = np.maximum(G - Keff, 0.0) if option_type == 'call' else np.maximum(Keff - G, 0.0)
        geo_exact = _black_on_forward(Fgeo, Keff, np.sqrt(vG2), 1.0, option_type)
        x = pay_a - pay_g
        est = x.mean() + geo_exact
    else:
        x = pay_a
        est = x.mean()
    if antithetic:
        h = M // 2
        pairs = 0.5 * (x[:h] + x[h:])
        se = pairs.std(ddof=1) / np.sqrt(h)
    else:
        se = x.std(ddof=1) / np.sqrt(len(x))
    return (D * est, D * se) if return_se else D * est

def solve_zero_cost_put(t, struct, f1, f2, smile1, smile2, rho, past_fix=None, D=1.0, method='mm', **kw):
    args = dict(f1=f1, f2=f2, smile1=smile1, smile2=smile2, rho=rho,
                method=method, past_fix=past_fix, D=D, **kw)
    C = price_asian_basket(t, struct, struct.K_call, option_type='call', **args)
    Fbar = basket_forward(struct, t, f1, f2, past_fix)
    h = lambda K: price_asian_basket(t, struct, K, option_type='put', **args) - C
    lo, hi = 0.2 * Fbar, 3.0 * Fbar
    if h(lo) * h(hi) > 0:
        raise ValueError("Check call price and smiles.")
    return brentq(h, lo, hi, xtol=1e-8)

t0 = pd.Timestamp("2020-01-06")
f1_0, f2_0, s1, s2, rho_0 = 60.0, 60.8, 0.38, 0.35, 0.97

p = black_76(60., 65., 0.4, 0.0, 1.0, 'call')
assert abs(implied_volatility(60., 65., 1.0, 0.0, p, 'call') - 0.4) < 1e-6
print("1) Black-76 / IV : OK")

Kp = solve_zero_cost_put(t0, struct, f1_0, f2_0, Smile([50, 60, 70], [s1]*3, f1_0), Smile([50, 60, 70], [s2]*3, f2_0), rho_0)
sm1, sm2 = Smile([50, 60, 70], [s1]*3, f1_0), Smile([50, 60, 70], [s2]*3, f2_0)
C = price_asian_basket(t0, struct, 78.0, f1_0, f2_0, sm1, sm2, rho_0, 'call')
P = price_asian_basket(t0, struct, Kp, f1_0, f2_0, sm1, sm2, rho_0, 'put')
assert abs(C - P) < 1e-6
print("2) Kput = %.3f ; Call(78) = %.4f = Put(Kput) = %.4f : OK" % (Kp, C, P))